In [1]:
import os
from dotenv import load_dotenv

print("loaded:", load_dotenv(r"API.env"))   

loaded: True


# Stage A: summary generation & LLM parsing

Choosen 18 RTLs to fine tune the prompts on to save token

## technical summary generation of the selected modules

In [18]:
#config
from pathlib import Path
import json, re, time, hashlib, os

SPEC_PDF  = Path("NEORV32-patched.pdf")
RTL_DIR   = Path("RTL_data")
OUT_DIR   = Path("summaries_tuning18"); OUT_DIR.mkdir(exist_ok=True)
CACHE_DIR = Path("rag_cache_ada002");   CACHE_DIR.mkdir(exist_ok=True)

CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 1000, 200, 20   # paper params
EMBED_MODEL = "text-embedding-ada-002"             # paper's embedder
GEN_MODEL   = "gpt-5-mini"                         # paper: GPT-5 family

QUERY_RULE = "name+filelist-role-v1"

# ===================== ABLATION SWITCH =====================
# Picks which (core prompt, ICL examples) pair Stage B uses. The registry lives in the
# Stage B cell -- add an entry there, set VERSION here, re-run Stage B.
#   v0  P3164-faithful examples, deliberately NOT format-matched (icl_examples_0.py)
#   v1  adapted examples: our parsed shape + our output contract (icl_asset_examples.py)
#   v01 worked triples: omsp_gpio + tiny_aes           (icl_examples_01.py)
#   v02 same GPIO, AES = aes_highthroughput_lowarea    (icl_examples_02.py)
# Core-prompt variants, all on the v01 ICL block:
#   v01c2  A-02  + "role, not location"                 baseline v01,   3 repeats
#   v01c3  A-03  = v01c2 + port-record granularity      baseline v01c2, 3 repeats
#   v01r   A-03 control = v0 core + the same A-03 edits baseline v01,   2 repeats
#
# STANDING BASELINE -- v01c5, promoted 2026-08-07.
#
# THE CHAIN IS INTACT: every hop moves exactly one thing, so this promotion disturbs no
# earlier arm and nothing needs re-running.
#     v0    -> v01     ICL block                    A-01
#     v01   -> v01c2   core, role-not-location      A-02
#     v01c2 -> v01c3   core, port granularity       A-03  (settled 5v5: precision +0.050
#                      [+0.021,+0.080], F1 +0.060 [+0.027,+0.091]; the port-recall guard
#                      scare was an artefact of 5 repeats compared against 3)
#     v01c3 -> v01c4   core, STEP 1 exposed         T-1
#     v01c4 -> v01c5   ICL now shows concepts       measured NULL on everything --
#                      recall +0.004, precision +0.009, F1 +0.011. The example edit bought
#                      nothing; it was built for a hypothesis that turned out to be wrong.
#     v01c5 -> v01c6   core, per-concept closure sweep     A-08
#     v01c6 -> v01c6p1 summary dropped everywhere          P-1  (null on all three)
#     v01c6p1 -> v01c6p1p2  parsed closed set dropped      P-2
#
# NEVER compare v01c5 to v01c3 directly -- that is two variables. Go through v01c4.
#
# FOOTNOTE on v01c3 -> v01c4: it changed the output CONTRACT, not just wording. Asking a
# model to write out reasoning it previously kept internal can alter the reasoning itself,
# so that step generalises less far than a pure wording change would. A caveat on
# interpretation, not a defect in the measurement.
#
# A new arm is CURRENT_BASELINE plus ONE change and pairs against it. Add it to BASELINE in
# the ablation cell, or the default below sends it to CURRENT_BASELINE -- either way it
# never falls through to a multi-variable contrast against VERSIONS[0].
CURRENT_BASELINE = "v01c6p1"

VERSION = "v01c6p1p2"

# Repeats of the SAME prompt. Generation is nondeterministic, so one run per version
# cannot separate a real prompt effect from sampling noise. The spread across repeats is
# the noise floor that every version difference has to clear before it means anything;
# 3 is the practical minimum for a usable sd.
# v01r was the A-03 control and ran at 2 -- reset to 3 afterwards. A 2-repeat arm has no
# usable sd and is directional only; do not leave this at 2 by accident.
REPEATS = 3

def asset_dir(version=None, rep=0, create=False):
    """Path for one (version, repeat). Nothing is ever overwritten.

    create=False by default: merely naming a directory must not bring it into existence.
    An empty assets_tuning18_<v>_r0 sitting on disk shadows a real earlier run during
    evaluation and scores as all-zeros. Only the generation loop passes create=True.
    """
    d = Path(f"assets_tuning18_{version or VERSION}_r{rep}")
    if create:
        d.mkdir(exist_ok=True)
    return d

print(f"VERSION = {VERSION}   REPEATS = {REPEATS}   -> "
      f"assets_tuning18_{VERSION}_r0..r{REPEATS - 1}")


VERSION = v01c6p1p2   REPEATS = 3   -> assets_tuning18_v01c6p1p2_r0..r2


In [3]:
from pathlib import Path

def discover_modules(rtl_dir=RTL_DIR):
    """All .vhd files in the RTL dir — the full 18-module tuning set.
    Only the filename stem (= IP module name) is used at this stage;
    RTL content is not read until Task 3 (Modular RTL Parsing)."""
    return [(p.stem, p) for p in sorted(Path(rtl_dir).glob("*.vhd"))]

modules = discover_modules()
assert len(modules) == 18, f"expected 18, found {len(modules)}"
print([s for s, _ in modules])

['neorv32_boot_rom', 'neorv32_bus', 'neorv32_cache', 'neorv32_cpu', 'neorv32_cpu_cp_cfu', 'neorv32_cpu_cp_muldiv', 'neorv32_cpu_pmp', 'neorv32_debug_dtm', 'neorv32_fifo', 'neorv32_hwspinlock', 'neorv32_imem', 'neorv32_package', 'neorv32_spi', 'neorv32_sys', 'neorv32_trng', 'neorv32_twi', 'neorv32_uart', 'neorv32_wdt']


In [4]:
#Cell 1 — PDF extraction + patch sanity gate
import fitz  # pymupdf

def extract_pages(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for i, page in enumerate(doc):
        txt = page.get_text("text")
        txt = re.sub(r"[ \t]+", " ", txt)
        txt = re.sub(r"\n{3,}", "\n\n", txt)
        pages.append({"page": i + 1, "text": txt})
    doc.close()
    return pages

pages = extract_pages(SPEC_PDF)
print(f"{len(pages)} pages, {sum(len(p['text']) for p in pages):,} chars")

239 pages, 419,345 chars


In [5]:
FOOTER_RE = re.compile(
    # page header
    r"The\s+NEORV32\s+RISC-V\s+Processor\s+Visit\s+on\s+GitHub"
    r"|"
    # page footer: pageno / 238 + (Version ... | Copyright ...) + date, as ONE unit
    r"\d{1,3}\s*/\s*238\s+"
    r"(?:Version\s+v1\.11\.0-r51-gd29fe6ec(?:-patched)?"
    r"|Copyright\s+by\s+Stephan\s+Nolting\.\s+All\s+rights\s+reserved\.)"
    r"\s+\d{4}-\d{2}-\d{2}"
)

pages_raw = extract_pages(SPEC_PDF)
assert any("-patched" in p["text"] for p in pages_raw), "wrong PDF: no -patched marker"

pages = [{"page": p["page"], "text": FOOTER_RE.sub(" ", p["text"]).strip()}
         for p in pages_raw]

# quick residue check — should print nothing:
for p in pages:
    if "Visit on GitHub" in p["text"] or re.search(r"/\s*238", p["text"]):
        print("footer residue on page", p["page"])

In [6]:
#cell 2— page aware chunking
def chunk_pages(pages, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks, step = [], size - overlap
    for p in pages:
        txt = p["text"].strip()
        if not txt:
            continue
        for start in range(0, max(len(txt) - overlap, 1), step):
            piece = txt[start:start + size].strip()
            if len(piece) < 100:
                continue
            chunks.append({"id": len(chunks), "page": p["page"], "text": piece})
    return chunks

EXCLUDE_PAGES = {1, 2, 3, 4, 5, 6, 7,        # cover, doc links, TOC
                 8, 12, 13,                   # ch.1 overview / marketing bullets
                 9, 10, 11,                   # §1.1 Rationale — author essay, no spec content
                 192, 193,                    # §4.2 HAL driver-file table (both pages)
                 210,                         # §4.10 RTE — software trap handling
                 237, 238, 239}               # ch.6 Legal (all three pages)                        # only pages verified as cover/TOC/index/name-lists.
                        # Do NOT exclude content pages that merely rank too often.
chunks = chunk_pages([p for p in pages if p["page"] not in EXCLUDE_PAGES])

print(f"{len(chunks)} chunks")

495 chunks


In [7]:
# Derive per-module role lines mechanically from the datasheet's own RTL File List (p.15-16)
role_src = " ".join(p["text"] for p in pages_raw if p["page"] in (15, 16))
role_src = re.sub(r"\s+", " ", role_src)
ROLE_RE = re.compile(r"neorv32_(\w+?)\.vhd\s*-\s*(.+?)(?=\s*[│├└]?\s*-?\s*neorv32_\w+\.vhd|$)")
FILE_ROLES = {f"neorv32_{m.group(1)}": m.group(2).strip(" -│├└")
              for m in ROLE_RE.finditer(role_src)}

# verify coverage for the 18 tuning modules before trusting it:
for stem, _ in modules:
    print(f"{stem:24s} -> {FILE_ROLES.get(stem, '*** MISSING ***')}")

neorv32_boot_rom         -> Bootloader ROM
neorv32_bus              -> SoC bus infrastructure modules
neorv32_cache            -> Generic cache module
neorv32_cpu              -> NEORV32 CPU TOP ENTITY
neorv32_cpu_cp_cfu       -> Custom instructions co-processor (Zxcfu ext.)
neorv32_cpu_cp_muldiv    -> Mul/Div co-processor (M ext.)
neorv32_cpu_pmp          -> Physical memory protection unit (Smpmp ext.)
neorv32_debug_dtm        -> On-chip debugger: debug transfer module
neorv32_fifo             -> Generic FIFO component
neorv32_hwspinlock       -> *** MISSING ***
neorv32_imem             -> Generic processor-internal instruction memory The NEORV32 RISC-V Processor Visit on GitHub 14 / 238 Version v1.11.0-r51-gd29fe6ec-patched 2026-07-20
neorv32_package          -> Main VHDL package file
neorv32_spi              -> Serial peripheral interface controller (SPI host)
neorv32_sys              -> System infrastructure modules
neorv32_trng             -> True random number generator
neorv32_t

In [8]:
#Cell 3 — ada-002 embeddings + FAISS (cached)
import numpy as np, faiss
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
oai = OpenAI()  # reads OPENAI_API_KEY

INDEX_PATH  = CACHE_DIR / "spec.faiss"
CHUNKS_PATH = CACHE_DIR / "chunks.json"
HASH_PATH   = CACHE_DIR / "index_hash.txt"
# Cell 3, replace the index_hash line:
CLEAN_TAG = "footerstrip-v1"   # bump whenever extraction/cleaning logic changes
chunks_digest = hashlib.sha256(
    "".join(c["text"] for c in chunks).encode()
).hexdigest()[:16]
index_hash = hashlib.sha256(
    f"{chunks_digest}|{CHUNK_SIZE}|{CHUNK_OVERLAP}|{EMBED_MODEL}|{CLEAN_TAG}".encode()
).hexdigest()[:16]

def embed_texts(texts, batch=128):
    """ada-002, batched. Returns float32 array, L2-normalized (IP == cosine)."""
    vecs = []
    for i in range(0, len(texts), batch):
        resp = oai.embeddings.create(model=EMBED_MODEL, input=texts[i:i + batch])
        vecs.extend(d.embedding for d in resp.data)
        print(f"  embedded {min(i + batch, len(texts))}/{len(texts)}")
    arr = np.asarray(vecs, dtype="float32")
    arr /= np.linalg.norm(arr, axis=1, keepdims=True)
    return arr

if INDEX_PATH.exists() and HASH_PATH.exists() and HASH_PATH.read_text() == index_hash:
    index = faiss.read_index(str(INDEX_PATH))
    chunks = json.loads(CHUNKS_PATH.read_text())
    print(f"loaded cached index ({index.ntotal} vectors)")
else:
    embs = embed_texts([c["text"] for c in chunks])
    index = faiss.IndexFlatIP(embs.shape[1])          # 1536-dim
    index.add(embs)
    faiss.write_index(index, str(INDEX_PATH))
    CHUNKS_PATH.write_text(json.dumps(chunks))
    HASH_PATH.write_text(index_hash)
    print(f"built index: {index.ntotal} vectors, hash {index_hash}")

def retrieve(query, k=TOP_K):
    q = embed_texts([query])
    scores, ids = index.search(q, k)
    return [{**chunks[i], "score": float(s)} for i, s in zip(ids[0], scores[0])]

loaded cached index (495 vectors)


In [ ]:
#Cell 5 — prompts (revised: name-only identity and query)
import prompts, importlib
importlib.reload(prompts)  # for iterative dev, reload the module
from prompts import SUMMARY_SYSTEM

def build_query(stem):
    """User-guided query: module name + the datasheet's own RTL File List
    one-line description (§1.4). Uniform mechanical rule, no per-module tuning."""
    human = stem.replace("neorv32_", "").replace("_", " ")
    role = FILE_ROLES.get(stem, "")
    return f"NEORV32 {human} module {role}".strip()

def build_summary_user(stem, retrieved):
    excerpts = "\n\n".join(
        f"[Excerpt {i+1} | p.{c['page']}]\n{c['text']}"
        for i, c in enumerate(retrieved)
    )
    return (f"TARGET IP MODULE: {stem}\n\n"
            f"=== RETRIEVED SPECIFICATION EXCERPTS ===\n{excerpts}\n"
            f"=== END OF EXCERPTS ===\n\n"
            f"Produce the Technical Summary for IP module '{stem}' now, "
            f"following the required structure.")

In [ ]:
for stem in ["neorv32_cpu_pmp", "neorv32_cpu", "neorv32_debug_dtm", "neorv32_package"]:
    got = sorted({c["page"] for c in retrieve(build_query(stem))})
    print(f"{stem:22s} pages={got}   CSR-region hits={[p for p in got if 160<=p<=175]}")

In [ ]:
#Cell 6 — generation loop (revised: OpenAI, gpt-5-mini)
# `oai` client already created in Cell 3 for ada-002.

def generate(system, user, retries=4):
    for attempt in range(retries):
        try:
            resp = oai.chat.completions.create(
                model=GEN_MODEL,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}],
                # NOTE: gpt-5-family models may reject non-default temperature;
                # omit it rather than pin it. Log the default in provenance.
            )
            return resp.choices[0].message.content
        except Exception as e:
            wait = 20 * (attempt + 1)
            print(f"  retry {attempt+1}: {e} (sleep {wait}s)")
            time.sleep(wait)
    raise RuntimeError("generation failed after retries")

for stem, _vhd_path in modules:
    out_md = OUT_DIR / f"{stem}.md"
    if out_md.exists():
        print(f"skip {stem} (exists)")
        continue
    query = build_query(stem)
    retrieved = retrieve(query)
    summary = generate(SUMMARY_SYSTEM, build_summary_user(stem, retrieved))
    out_md.write_text(summary, encoding="utf-8")
    (OUT_DIR / f"{stem}.provenance.json").write_text(json.dumps({
        "module": stem,
        "query": query,
        "query_rule": QUERY_RULE,          # <-- here
        "file_role": FILE_ROLES.get(stem, ""),          # <-- optional, see below
        "chunk_ids": [c["id"] for c in retrieved],
        "pages": sorted({c["page"] for c in retrieved}),
        "scores": [round(c["score"], 4) for c in retrieved],
        "embed_model": EMBED_MODEL, "gen_model": GEN_MODEL,
        "index_hash": index_hash, "spec": SPEC_PDF.name,
    }, indent=2), encoding="utf-8")
    print(f"done {stem}  pages={sorted({c['page'] for c in retrieved})}")

### checkings

In [ ]:
from collections import Counter

prov = [json.loads(p.read_text()) for p in sorted(OUT_DIR.glob("*.provenance.json"))]
freq = Counter(pg for r in prov for pg in r["pages"])
attractors = [pg for pg, n in freq.most_common() if n >= 8]
print("attractor pages (>=8/18 modules):", attractors)

for pg in attractors:
    txt = next(p["text"] for p in pages if p["page"] == pg)
    print(f"\n=== p.{pg} (in {freq[pg]}/18) ===\n{txt[:300]}")

In [ ]:
#Cell 7 — QC, aimed at the fine-tuning objective
# Check both ends of the spectrum:
#   asset-rich modules -> section 2/5 must name real registers/CSRs
#   asset-poor modules -> section 5 must NOT be inflated
QC_RICH = ["neorv32_cpu_pmp", "neorv32_trng", "neorv32_wdt", "neorv32_fifo", "neorv32_package", "neorv32_hwspinlock"]
QC_POOR = [s for s, _ in modules if s not in QC_RICH][:3]  # eyeball a few others

for stem in QC_RICH + QC_POOR:
    f = OUT_DIR / f"{stem}.md"
    if f.exists():
        print(f"\n{'='*60}\n{stem}\n{'='*60}\n{f.read_text()[:1500]}")



In [ ]:
t = (OUT_DIR / "neorv32_cpu_pmp.md").read_text()
print("D3 content present:", "31 downto 2" in t or "downto 2" in t)

In [ ]:
# Registers/ports owned by module X must not appear in module Y's summary.
FOREIGN = {"neorv32_spi": "SPI_", "neorv32_twi": "TWI_", "neorv32_uart": "UART_",
           "neorv32_wdt": "WDT_", "neorv32_trng": "TRNG_",
           "neorv32_cpu_pmp": "PMPCFG_", "neorv32_cpu_cp_cfu": "cfureg"}

for md in sorted(OUT_DIR.glob("*.md")):
    text = md.read_text(encoding="utf-8")
    for owner, prefix in FOREIGN.items():
        if md.stem != owner and prefix in text:
            line = next(l for l in text.splitlines() if prefix in l)
            print(f"CHECK  {prefix!r} in {md.stem:24s} | {line.strip()}")

In [ ]:
SENTINEL = "Not covered in retrieved spec sections."
LEAK_TERMS = ["excerpt", "retriev", "provided context", "the context above",
              "these instructions", "system prompt", "[Excerpt"]

leaks = []
for md in sorted(OUT_DIR.glob("*.md")):
    body = md.read_text(encoding="utf-8").replace(SENTINEL, "")  # whitelist sentinel
    low = body.lower()
    for term in LEAK_TERMS:
        if term.lower() in low:
            # show the offending line for eyeball
            line = next(l for l in body.splitlines() if term.lower() in l.lower())
            leaks.append((md.stem, term, line.strip()[:120]))

if leaks:
    for stem, term, line in leaks:
        print(f"LEAK  {stem:24s} [{term}]  {line}")
else:
    print("meta-leakage: clean (18/18)")

In [ ]:
import re

EXPECTED = ["1. FUNCTION AND ROLE",
            "2. REGISTERS, CSRS, AND FLAGS",
            "3. CONFIGURATION",
            "4. CROSS-MODULE INTERACTION",
            "5. SECURITY-RELEVANT BEHAVIOR"]

fails = []
for md in sorted(OUT_DIR.glob("*.md")):
    body = md.read_text(encoding="utf-8")
    if not body.lstrip().startswith(f"MODULE: {md.stem}"):
        fails.append((md.stem, "MODULE header missing or wrong name"))
    found = re.findall(r"^\s*(\d\. [A-Z][A-Z ,\-]+)\s*$", body, re.MULTILINE)
    found = [f.strip() for f in found]
    if found != EXPECTED:
        fails.append((md.stem, f"sections={found}"))
    # every section must have content or the exact sentinel — no empty sections
    for i, hdr in enumerate(EXPECTED):
        start = body.find(hdr) + len(hdr)
        end = body.find(EXPECTED[i+1]) if i+1 < len(EXPECTED) else len(body)
        if not body[start:end].strip():
            fails.append((md.stem, f"empty section: {hdr}"))

if fails:
    for stem, msg in fails:
        print(f"FAIL  {stem:24s} {msg}")
else:
    print("section structure: clean (18/18)")

In [ ]:
full = "\n".join(p["text"] for p in pages_raw)   # pre-exclusion, whole document
for term in ["hwspinlock", "spinlock", "neorv32_fifo", "Hardware Spinlock"]:
    hits = [p["page"] for p in pages_raw if term.lower() in p["text"].lower()]
    print(f"{term!r:22s} pages={hits}")

In [ ]:
import re, json
from pathlib import Path

full_squash = re.sub(r"\s+", "", "".join(p["text"] for p in pages_raw)).lower()
by_id = {c["id"]: c["text"] for c in chunks}

TOKEN_RES = [
    r"\b[A-Z][A-Z0-9]+(?:_[A-Z0-9]+)+\b",            # WDT_CTRL_EN, PMP_NUM_REGIONS
    r"\b0x[0-9a-fA-F]{4,}\b",                        # 0xfffb0000, 0x709D1AB3
    r"\b[a-z][a-z0-9]*(?:_[a-z0-9]+)+_[io]\b",       # spi_csn_o, rstn_wdt_o
    r"\b(?:pmpcfg|pmpaddr|cfureg)\d+\b",
    r"\b(?:misa|mxisa|mhartid|mepc|dpc|mcounteren|dtmcs|idcode)\b",
]
TOKEN_RES += [
    r"\b[a-z][a-z0-9]*(?:_[a-z0-9]+)+_[a-z]\b",   # bus_req_t, bus_rsp_t
    r"\b[id]_bus_\*",                             # i_bus_*, d_bus_*
    r"\b(?:XBUS|SYSINFO|BOOTROM|IMEM|DMEM|CLINT|CFS|CFU|SLINK|NEOLED)\b",
    r"\b[id]CACHE\b", r"\bSmpmp\b", r"\bZ[a-z]{2,}\b",   # Zicsr, Zmmul, Zxcfu
    r"\b0b[01]+\b", r"\b\d+\s?(?:kB|MB|bit|-bit)\b",
]

report = {}
for md in sorted(OUT_DIR.glob("*.md")):
    stem = md.stem
    prov = json.loads((OUT_DIR / f"{stem}.provenance.json").read_text())
    ctx = re.sub(r"\s+", "", " ".join(by_id[i] for i in prov["chunk_ids"])).lower()
    body = md.read_text(encoding="utf-8")

    toks = set()
    for rx in TOKEN_RES:
        toks |= set(re.findall(rx, body))
    grounded, ungrounded_but_real, hallucinated = [], [], []
    for t in sorted(toks):
        k = re.sub(r"\s+", "", t).lower()
        if k in ctx:                grounded.append(t)
        elif k in full_squash:      ungrounded_but_real.append(t)
        else:                       hallucinated.append(t)

    report[stem] = dict(n=len(toks), grounded=len(grounded),
                        leaked=ungrounded_but_real, halluc=hallucinated)
    flag = "" if not (ungrounded_but_real or hallucinated) else "  <-- REVIEW"
    print(f"{stem:24s} {len(grounded):3d}/{len(toks):3d} grounded{flag}")
    if ungrounded_but_real: print(f"    LEAKED (in spec, not in retrieved ctx): {ungrounded_but_real}")
    if hallucinated:        print(f"    NOT IN SPEC AT ALL:                     {hallucinated}")

tot = sum(r["n"] for r in report.values()); g = sum(r["grounded"] for r in report.values())
print(f"\ncorpus grounding rate: {g}/{tot} = {100*g/tot:.1f}%")

In [ ]:
prov = json.loads((OUT_DIR/"neorv32_cpu.provenance.json").read_text())
for i in prov["chunk_ids"]:
    if "dbus" in by_id[i].lower():
        print(i, [w for w in by_id[i].split() if "dbus" in w.lower()])

In [ ]:
from pathlib import Path
import re

A = {p.stem for p in Path("RTL_data").glob("*.vhd")}                 # LAsset RTL (51)
RAW_57 = """
$NEORV32_HOME/rtl/core/neorv32_package.vhd
...paste all 57 lines...
"""
B = {Path(l.strip()).stem for l in RAW_57.strip().splitlines() if l.strip()}
C = set(re.findall(r"(neorv32_\w+)\.vhd",                            # datasheet §1.4 @ d29fe6ec
                   " ".join(p["text"] for p in pages_raw if p["page"] in (15, 16))))

print(f"A LAsset RTL_data      : {len(A)}")
print(f"B pasted list          : {len(B)}")
print(f"C datasheet @ d29fe6ec : {len(C)}\n")

def show(label, s):
    print(f"{label} ({len(s)}):")
    for x in sorted(s): print("   ", x)
    print()

show("A \\ C  — in your RTL, spec silent  [THE ONE THAT MATTERS]", A - C)
show("C \\ A  — spec documents, RTL absent", C - A)
show("A \\ B", A - B)
show("B \\ A", B - A)

# rename detection across naming conventions (boot_rom vs bootrom, cp_ vs alu_)
def norm(s): return s.replace("_", "").lower()
nA, nB = {norm(x): x for x in A}, {norm(x): x for x in B}
for k in set(nA) & set(nB):
    if nA[k] != nB[k]: print(f"RENAME: {nA[k]} <-> {nB[k]}")

# version stamp carried in the RTL itself
pkg = Path("RTL_data/neorv32_package.vhd").read_text(errors="replace")
for line in pkg.splitlines():
    if re.search(r"(?i)version", line): print("PKG:", line.strip())

## RTL parsing

In [9]:
# Cell 3 — the client (OpenAI; OPENAI_API_KEY in env)
import os, time, random, json
from openai import OpenAI

class LLMClient:
    def __init__(self, model=GEN_MODEL, effort="medium", max_retries=6):
        self.c = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        self.model, self.effort, self.r = model, effort, max_retries
        self.usage = []            # per-call token accounting for cost logging

    def _call(self, system, user, json_mode, max_tokens, effort):
        kw = dict(
            model=self.model,
            instructions=system,
            input=user,
            reasoning={"effort": effort or self.effort},
            max_output_tokens=max_tokens,
        )
        if json_mode:
            kw["text"] = {"format": {"type": "json_object"}}

        wait = 2.0
        for a in range(self.r):
            try:
                r = self.c.responses.create(**kw)
                if getattr(r, "status", None) == "incomplete":
                    # ran out of max_output_tokens (reasoning shares the budget):
                    # surface it, return the (possibly truncated) text and let the
                    # caller's escalation path retry with a bigger budget.
                    why = getattr(getattr(r, "incomplete_details", None), "reason", "?")
                    print(f"    [llm warn] incomplete response ({why}) at max_output_tokens={max_tokens}")
                u = r.usage
                self.usage.append({
                    "model": self.model,
                    "in": u.input_tokens,
                    "out": u.output_tokens,
                    "reasoning": u.output_tokens_details.reasoning_tokens,
                    "cached_in": getattr(u.input_tokens_details, "cached_tokens", 0),
                })
                return r.output_text
            except Exception as e:
                if a == self.r - 1:
                    raise
                time.sleep(wait + random.uniform(0, 1)); wait *= 2

    def complete(self, system, user, max_tokens=4096, effort=None):
        """JSON-mode call (Stage A batches, asset generation)."""
        return self._call(system, user, True, max_tokens, effort)

    def complete_text(self, system, user, max_tokens=4096, effort=None):
        """Plain-text call (markdown technical summaries)."""
        return self._call(system, user, False, max_tokens, effort)

    def cost(self, price_in=0.25, price_out=2.00, price_cached=0.025):
        """USD for calls so far, at per-1M-token prices (default gpt-5-mini)."""
        i = sum(u["in"] - u["cached_in"] for u in self.usage)
        c = sum(u["cached_in"] for u in self.usage)
        o = sum(u["out"] for u in self.usage)
        rt = sum(u["reasoning"] for u in self.usage)
        total = (i*price_in + c*price_cached + o*price_out) / 1e6
        return {"calls": len(self.usage), "in": i, "cached_in": c,
                "out": o, "reasoning": rt, "usd": round(total, 4)}

client = LLMClient()

In [10]:
# ===================== Cell — Modular RTL Parsing (Prm_m) =====================
import importlib, prompts, rtl_parse
importlib.reload(prompts); importlib.reload(rtl_parse)
from prompts import PARSE_PORTS_ANNOTATE_SYSTEM, PARSE_SIGNALS_ANNOTATE_SYSTEM
from rtl_parse import build_record_registry, parse_rtl_file

# package + all modules -> so bus_req_t / bus_rsp_t (declared in neorv32_package) are known everywhere
RECORDS = build_record_registry(sorted(RTL_DIR.glob("*.vhd")))
print(f"{len(RECORDS)} record types; bus_req_t={'bus_req_t' in RECORDS}, bus_rsp_t={'bus_rsp_t' in RECORDS}")

PARSE_DIR = Path("parsed_tuning18"); PARSE_DIR.mkdir(exist_ok=True)

def _loads(txt):
    try:
        return json.loads(txt)
    except Exception:
        pass
    m = re.search(r"\{.*\}", txt, re.S)   # salvage a complete object if the tail was cut
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    return {}                              # give up gracefully rather than raise


28 record types; bus_req_t=True, bus_rsp_t=True


In [11]:
def _annot_user(stem, entity, rtl, elems, key):
    return (f"MODULE: {stem}\nENTITY: {entity}\n\n"
            f"AUTHORITATIVE {key.upper()} of entity '{entity}' (ground truth from RTL — annotate "
            f"all, add/drop none; names include record fields written verbatim with a dot, "
            f"e.g. 'ctrl.buf_req', 'host_req_i.stb'):\n{json.dumps(elems, indent=2)}\n\n"
            f"VHDL SOURCE:\n{rtl}\n\n"
            f"Return the annotated {key} as a JSON object using the required schema.")

def _annotate(system, stem, entity, rtl, elems, key, batch=50, max_tokens=8192):
    """Annotate in batches so large record-expanded port lists (e.g. bus io_switch = 596
    ports) don't overflow the model's output and truncate the JSON. A batch that still
    fails to parse degrades its elements to 'unclear' instead of crashing the module."""
    ann = {}
    for i in range(0, len(elems), batch):
        chunk = elems[i:i + batch]
        got = _loads(client.complete(system, _annot_user(stem, entity, rtl, chunk, key),
                                     max_tokens=max_tokens)).get(key, [])
        for a in got:
            if "name" in a:
                ann[a["name"]] = a
        if not got:
            print(f"    [annot warn] {stem}/{entity} {key} batch {i//batch}: "
                  f"{len(chunk)} elems left unclear")
    return ann

def annotate_ports(stem, entity, rtl, ports):
    a = _annotate(PARSE_PORTS_ANNOTATE_SYSTEM, stem, entity, rtl, ports, "ports")
    return {n: v.get("function", "unclear from RTL") for n, v in a.items()}

def annotate_signals(stem, entity, rtl, signals):
    a = _annotate(PARSE_SIGNALS_ANNOTATE_SYSTEM, stem, entity, rtl, signals, "signals")
    return {n: {"kind": v.get("kind", "signal"),
                "function": v.get("function", "unclear from RTL")} for n, v in a.items()}

def parse_module(stem, path, records, overwrite=False):
    outp = PARSE_DIR / f"{stem}.json"
    if outp.exists() and not overwrite:
        return json.loads(outp.read_text())
    d = parse_rtl_file(path, records=records)
    rtl = Path(path).read_text(encoding="utf-8", errors="ignore")
    ports_all, signals_all = [], []
    for ent in d["entities"]:                       # annotate per entity -> no cross-entity name clashes
        en = ent["entity"]
        pf = annotate_ports(stem, en, rtl, ent["ports"])
        sf = annotate_signals(stem, en, rtl, ent["signals"])
        for p in ent["ports"]:
            ports_all.append({"entity": en, **p, "function": pf.get(p["name"], "unclear from RTL")})
        for s in ent["signals"]:
            signals_all.append({"entity": en, **s,
                                "kind": sf.get(s["name"], {}).get("kind", "signal"),
                                "function": sf.get(s["name"], {}).get("function", "unclear from RTL")})
    parsed = {"module": stem, "entities": [e["entity"] for e in d["entities"]],
              "ports": ports_all, "signals": signals_all}
    outp.write_text(json.dumps(parsed, indent=2))
    return parsed

parsed_all = {}
for stem, path in modules:
    try:
        parsed_all[stem] = parse_module(stem, path, RECORDS)
        p = parsed_all[stem]
        print(f"[ok  ] {stem:24s} entities={p['entities']}  {len(p['ports'])}p / {len(p['signals'])}s")
    except Exception as e:
        print(f"[err ] {stem:24s} {e}")

[ok  ] neorv32_boot_rom         entities=['neorv32_boot_rom']  19p / 2s
[ok  ] neorv32_bus              entities=['neorv32_bus_switch', 'neorv32_bus_reg', 'neorv32_bus_gateway', 'neorv32_bus_io_switch', 'neorv32_bus_amo_rmw', 'neorv32_bus_amo_rvs']  811p / 59s
[ok  ] neorv32_cache            entities=['neorv32_cache', 'neorv32_cache_memory']  46p / 40s
[ok  ] neorv32_cpu              entities=['neorv32_cpu']  49p / 66s
[ok  ] neorv32_cpu_cp_cfu       entities=['neorv32_cpu_cp_cfu']  16p / 13s
[ok  ] neorv32_cpu_cp_muldiv    entities=['neorv32_cpu_cp_muldiv']  46p / 24s
[ok  ] neorv32_cpu_pmp          entities=['neorv32_cpu_pmp']  45p / 17s
[ok  ] neorv32_debug_dtm        entities=['neorv32_debug_dtm']  13p / 29s
[ok  ] neorv32_fifo             entities=['neorv32_fifo']  11p / 16s
[ok  ] neorv32_hwspinlock       entities=['neorv32_hwspinlock']  19p / 2s
[ok  ] neorv32_imem             entities=['neorv32_imem']  19p / 8s
[ok  ] neorv32_package          entities=[]  0p / 0s
[ok  ] neorv32

# Stage B: Asset generation

In [19]:
# ===================== Stage B: version registry + generation =====================
import importlib, prompts, rtl_parse, icl_asset_examples, icl_examples_0, icl_examples_01, icl_examples_02
import icl_examples_01b
import icl_examples_01b_nosum
import icl_examples_01b_noparse
import diagnose_step1 as DG
for m in (prompts, rtl_parse, icl_asset_examples, icl_examples_0, icl_examples_01,
          icl_examples_02, icl_examples_01b, icl_examples_01b_nosum,
          icl_examples_01b_noparse, DG):
    importlib.reload(m)

# Register one (core prompt, ICL examples) pair per version. To add a version:
# drop a new module next to these, import it above, add a line here, set VERSION
# in the config cell. Splice by CONCATENATION -- the examples contain literal JSON
# braces, so an f-string would need every brace escaped.
PROMPT_VERSIONS = {
    "v0": (prompts.ASSET_PRIMARY_CORE, icl_examples_0.ICL_ASSET_EXAMPLES_0),
    "v1": (prompts.ASSET_PRIMARY_CORE, icl_asset_examples.ICL_ASSET_EXAMPLES),
    # worked input->reasoning->output triple, omsp_gpio (LAsset RTL + Calgary manual labels)
    "v01": (prompts.ASSET_PRIMARY_CORE, icl_examples_01.ICL_ASSET_EXAMPLES_01),
    # same GPIO, AES swapped to aes_highthroughput_lowarea (4/5 P3164 blocks)
    "v02": (prompts.ASSET_PRIMARY_CORE, icl_examples_02.ICL_ASSET_EXAMPLES_02),
    # ---- A-02 -----------------------------------------------------------------
    # Same ICL block as v01; the CORE gains one insertion in STEP 2 ("role, not
    # location"). Everything else is byte-identical to v01, so a difference between
    # them is attributable to that insertion alone. Baseline for this arm is v01.
    "v01c2": (prompts.ASSET_PRIMARY_CORE_V2, icl_examples_01.ICL_ASSET_EXAMPLES_01),
    # ---- A-03 -----------------------------------------------------------------
    # Port-record granularity, ONE proposition applied in the two places the prompt
    # speaks about naming: the INPUTS example stops illustrating with a port-record
    # field ("host_req_i.stb" -> "rtx_engine.sreg"), and RULES gains the granularity
    # bullet. prompts._apply_a03() performs both edits and round-trip asserts that
    # nothing else moved, so the two arms below carry byte-identical A-03 text.
    #
    # v01c3 is the ARM       -- baseline v01c2, 3 repeats.
    # v01r  is the CONTROL   -- baseline v01, 2 repeats, DIRECTIONAL ONLY (no sd at
    #   n=2). It exists to disambiguate a null on v01c3 between "the rule does
    #   nothing" and "V2's ask-BOTH wording masks it", and is run now rather than
    #   conditionally so both face the same model snapshot.
    "v01c3": (prompts.ASSET_PRIMARY_CORE_V3, icl_examples_01.ICL_ASSET_EXAMPLES_01),
    "v01r": (prompts.ASSET_PRIMARY_CORE_V0R, icl_examples_01.ICL_ASSET_EXAMPLES_01),
    # ---- T-1 (Stage C substage, DIAGNOSTIC -- not a recall arm) ------------------
    # v01c3 + STEP 1 made visible, plus the contract slot it needs. Splits each miss
    # into (a) never conceived vs (b) conceived-but-unbound. It changes the OUTPUT
    # CONTRACT and asks for reasoning that was internal, both of which can move
    # behaviour on their own -- so a recall delta here is CONFOUNDED and must never be
    # quoted as an ablation result. The diagnostic classification is the deliverable.
    "v01c4": (prompts.ASSET_PRIMARY_CORE_V4, icl_examples_01.ICL_ASSET_EXAMPLES_01),
    # ---- T-1b -------------------------------------------------------------------
    # Same core as v01c4; the ICL examples now SHOW a ConceptualAssets section. v01c4
    # asked for that array in the contract and got it in 0 of 90 files, while the
    # nested "Concept" field -- which the examples do demonstrate -- was adopted in 85
    # of 90. This makes the demonstration agree with the instruction, and is the direct
    # test of whether the examples were the blocker.
    "v01c5": (prompts.ASSET_PRIMARY_CORE_V4, icl_examples_01b.ICL_ASSET_EXAMPLES_01B),
    # ---- A-08 -------------------------------------------------------------------
    # Per-concept closure sweep. Same ICL as v01c5; the CORE gains one insertion
    # immediately before RULES. T-1 found 82.1% of misses are BINDING failures, and
    # specifically COMPLETENESS ones -- the covering concept had typically already
    # bound 5-6 elements and the model stopped one short. So this is a verification
    # pass over a list the model has already written, not a fourth restatement of
    # fan-out (which it already complies with at 3.63 elements per concept).
    "v01c6": (prompts.ASSET_PRIMARY_CORE_V5, icl_examples_01b.ICL_ASSET_EXAMPLES_01B),
    # ---- P-1 (input ablation 1 of 3) --------------------------------------------
    # Same ICL as v01c6; the CORE drops the TECHNICAL SUMMARY declaration and the three
    # instructions that tell the model to ground in it, and INPUT_BLOCKS drops the block
    # itself from the user message. 4 hunks, -173 chars, zero surviving references.
    # The ICL block ALSO loses its two summary blocks and the two reasoning lines that
    # cited them ("Summary 5: ...", "Summary section 5 says ..."). Leaving those would
    # show the model a worked procedure -- consult the summary -- that the ablated task
    # cannot execute, which is a confound rather than a removal.
    "v01c6p1": (prompts.ASSET_PRIMARY_CORE_P1,
                icl_examples_01b_nosum.ICL_ASSET_EXAMPLES_01B_NOSUM),
    # ---- P-2 (input ablation 2 of 3) --------------------------------------------
    # v01c6p1 plus ONE proposition: the model builds the closed set itself. The two
    # PARSED JSON blocks leave the user message and the examples; the core RE-SOURCES
    # the closed-set contract from the RTL's own entity/port and architecture-signal
    # declarations rather than deleting it. Every downstream instruction that says
    # "closed set" therefore stays literally true and byte-identical -- DEFINITIONS,
    # STEP 2, T-1's contract line, and the whole of A-08's sweep. That is why the
    # log's original reason for dropping P-2 no longer holds; see prompts.py.
    #
    # NOT a LAsset replication: Alg.1 line 5 always receives Prm_m, so unlike P-1
    # there is no published counterpart to check against. The question is ours --
    # the blocks are 58% of every user message and cost an LLM call per module to
    # build, so if the RTL alone suffices, stage 4 is overhead.
    "v01c6p1p2": (prompts.ASSET_PRIMARY_CORE_P1P2,
                  icl_examples_01b_noparse.ICL_ASSET_EXAMPLES_01B_NOPARSE),
    # ---- L-1 (remove evaluation-set identifiers) ---------------------------------
    # Same ICL block; the CORE stops quoting four real NEORV32 element names as its
    # naming examples. One of them, ctrl_i, is a ground-truth ANSWER in 10 of the 41
    # annotated modules -- A-03's rule literally says a record port carrying an asset
    # is named whole, 'ctrl_i'. Replacements are checked absent from all 1355 parsed
    # element names and all 218 ground-truth names.
    #
    # NOT a confound for P-1 or P-2: the identifiers entered at A-03 and sit in every
    # later version, baseline and arm alike, so they cancel in every paired delta
    # measured so far. What they threaten is the absolute recall quoted against the
    # paper, and any run on the 26 held-out modules -- 9 of which have ctrl_i as an
    # answer. The rule itself is unchanged; only the names are.
    "v01c6p1p2d": (prompts.ASSET_PRIMARY_CORE_P1P2D,
                   icl_examples_01b_noparse.ICL_ASSET_EXAMPLES_01B_NOPARSE),
}

_core, _icl = PROMPT_VERSIONS[VERSION]
ASSET_PRIMARY_SYSTEM = _core + "\n\n" + _icl

# Derived, never hand-maintained: if the composed prompt asks for the array, the run loop
# demands it back. A hand-kept list of "versions that emit concepts" is one edit away from
# the v01c4 failure, where a silent tolerance let 90 module-repeats pass while emitting none.
EXPECT_CONCEPTS = "ConceptualAssets" in ASSET_PRIMARY_SYSTEM

print(f"{VERSION}: core {len(_core)} + ICL {len(_icl)} = {len(ASSET_PRIMARY_SYSTEM)} chars"
      + ("   [expects ConceptualAssets]" if EXPECT_CONCEPTS else ""))


# Which user-message blocks a version receives. Absent version -> everything, so no
# existing version changes behaviour. The INPUT ABLATION family (P-1 / P-2' / P-3) asks
# whether line 5's over-emission is driven by input volume: it gets ~34.5k tokens per call
# (summary 11%, parsed ports 42%, parsed signals 16%, RTL 31%) and emits 428 assets for 111
# references, while LASP feeds a compact RTL abstraction because it cannot fit the RTL.
#
# Dropping a block here is only HALF the arm -- the core prompt declares its inputs and
# refers back to them, so each ablated version also pairs with a core variant that removes
# those references (prompts.ASSET_PRIMARY_CORE_P1 and friends). Leaving them would tell the
# model to ground in something absent, which is a confound rather than a removal.
INPUT_BLOCKS = {
    "v01c6p1": {"summary": False},                       # P-1
    "v01c6p1p2": {"summary": False, "parsed": False},    # P-2, on top of P-1
    "v01c6p1p2d": {"summary": False, "parsed": False},   # L-1, same inputs as P-2
}
# _ALL_BLOCKS is the DEFAULT, INPUT_BLOCKS is a sparse OVERRIDE. They do not conflict:
# build_asset_user merges {**_ALL_BLOCKS, **override} and later keys win, so v01c6p1
# resolves to summary=False with everything else left True. A version absent from
# INPUT_BLOCKS gets everything, so no existing version changes behaviour.
_ALL_BLOCKS = {"summary": True, "parsed": True, "rtl": True, "function": True}

# Two ways this could go wrong silently, both now checked at import.
#
# (1) A TYPO in an override key. {"summry": False} merges cleanly, leaves summary=True, and
#     the block is NOT dropped -- the arm would run as its own baseline and look like a
#     perfect null.
# (2) DRIFT between the dropped block and the prompt. If a version drops a block while its
#     core or ICL still declares it, the model is told to use something absent. That is a
#     confound, not an ablation -- and it is exactly what nearly wrecked P-1, whose ICL
#     examples still cited the summary by section after the core had stopped declaring it.
_BLOCK_MARKERS = {                       # (marker in core, marker in ICL examples)
    "summary":  ("TECHNICAL SUMMARY", "=== TECHNICAL SUMMARY ==="),
    "parsed":   ("PARSED I/O PORTS", "=== PARSED I/O PORTS (JSON) ==="),
    "rtl":      ("(4) RTL", "=== RTL ==="),
    "function": ("type, function", '"function":'),
}
for _v, _b in INPUT_BLOCKS.items():
    _unknown = set(_b) - set(_ALL_BLOCKS)
    assert not _unknown, (f"INPUT_BLOCKS[{_v!r}]: unknown block(s) {sorted(_unknown)} -- "
                          f"valid keys are {sorted(_ALL_BLOCKS)}. A typo here drops nothing.")
    assert _v in PROMPT_VERSIONS, f"INPUT_BLOCKS[{_v!r}] has no PROMPT_VERSIONS entry"
    _core, _icl = PROMPT_VERSIONS[_v]
    for _k, _keep in _b.items():
        if _keep:
            continue
        _cm, _im = _BLOCK_MARKERS[_k]
        assert _cm not in _core, (f"{_v} drops '{_k}' from the user message but its CORE "
                                  f"still declares it ({_cm!r}) -- confound, not ablation")
        assert _im not in _icl, (f"{_v} drops '{_k}' from the user message but its ICL "
                                 f"examples still demonstrate it ({_im!r}) -- the model would "
                                 f"be shown a procedure it cannot carry out")

# Also derived, and for the same reason. validate_primary checks every emitted name
# against the PARSED closed set, and the run loop reacts to a failure by regenerating
# once and keeping whichever attempt has fewer issues. For every version that receives
# the parsed blocks that retry is inert -- v01c6 and v01c6p1 both sit at 0 ungrounded
# names across all 90 module-repeats, so it has nothing to fire on.
#
# Under P-2 it would fire constantly, and that is not a cost, it is a LEAK: choosing
# the sample that agrees better with the parsed list is selection ON the closed set,
# using exactly the information the arm withholds from the prompt. The measurement
# would be of a two-sample filter, not of the model. So a version that drops the
# parsed blocks validates and REPORTS but does not regenerate.
#
# Derived from INPUT_BLOCKS rather than hand-listed: a hand-kept set of "versions
# that skip the retry" is one forgotten edit away from silently re-introducing the
# leak, which is the v01c4 failure mode in a different costume.
#
# DEFINED HERE, NOT BESIDE EXPECT_CONCEPTS. It reads INPUT_BLOCKS, which is declared
# above -- placing it earlier raises NameError in a fresh kernel and, in a kernel that
# has run this cell before, silently reads the PREVIOUS INPUT_BLOCKS and resolves to
# True. That failure leaves no trace in the output.
RETRY_ON_VALIDATION = INPUT_BLOCKS.get(VERSION, {}).get("parsed", True)
if not RETRY_ON_VALIDATION:
    print(f"{VERSION}: validation is REPORT-ONLY (no corrective regeneration) -- "
          f"this version does not receive the parsed closed set")


def build_asset_user(stem, summary, parsed, rtl, blocks=None):
    b = {**_ALL_BLOCKS, **(blocks or {})}
    ports, signals = parsed["ports"], parsed["signals"]
    if not b["function"]:                   # P-2': keep the closed set, drop the annotation
        ports = [{k: v for k, v in e.items() if k != "function"} for e in ports]
        signals = [{k: v for k, v in e.items() if k != "function"} for e in signals]
    parts = [f"TARGET IP MODULE: {stem}\n\n"]
    if b["summary"]:
        parts.append(f"=== TECHNICAL SUMMARY ===\n{summary}\n\n")
    if b["parsed"]:
        parts.append(f"=== PARSED I/O PORTS (JSON) ===\n{json.dumps(ports, indent=1)}\n\n")
        parts.append(f"=== PARSED INTERNAL SIGNALS (JSON) ===\n{json.dumps(signals, indent=1)}\n\n")
    if b["rtl"]:
        parts.append(f"=== RTL ===\n{rtl}\n\n")
    parts.append(f"Identify the primary security assets for '{stem}' and return the JSON object per the contract.")
    return "".join(parts)

def generate_assets(stem, path, parsed, out_dir, overwrite=False):
    """Cache ONLY parse-valid results.

    An unparseable response raises instead of being written, so a failed call can never
    masquerade as a genuine "this module has no assets" verdict -- which would silently
    become a free precision win for whichever prompt version happened to fail.
    """
    outp = Path(out_dir) / f"{stem}.json"
    if outp.exists() and not overwrite:
        return json.loads(outp.read_text())
    summary = (OUT_DIR / f"{stem}.md").read_text(encoding="utf-8")
    # Comments are stripped for the ASSET stage only. The asset prompt takes its
    # semantics from the technical summary and the parsed elements' `function` fields,
    # not from in-source comments -- so removing them drops ~30-60% of the RTL text
    # without losing anything the parser or the prompt relies on (parse_rtl_file returns
    # an identical closed set either way). Do NOT strip for the port/signal ANNOTATION
    # stage: that prompt is explicitly told to use in-source comments.
    rtl = rtl_parse.strip_comments(
        Path(path).read_text(encoding="utf-8", errors="ignore"),
        vhdl=Path(path).suffix.lower() in (".vhd", ".vhdl"))
    user = build_asset_user(stem, summary, parsed, rtl, INPUT_BLOCKS.get(VERSION))
    # max_output_tokens is a CEILING, not a reservation: you are billed for tokens the
    # model actually generates, so a high cap costs nothing on calls that finish early.
    # A single generous budget beats an escalation ladder, whose truncated first attempt
    # was discarded and paid for anyway (~12k wasted output tokens per escalation; at
    # effort="high" reasoning is ~83% of output, leaving little room for the JSON).
    txt = client.complete(ASSET_PRIMARY_SYSTEM, user, max_tokens=65536, effort="high")
    # Keep the response VERBATIM, before parsing, in a subdirectory that load_run() cannot
    # see (it globs "*.json" non-recursively, so a nested folder is invisible to scoring).
    # Costs disk and nothing else. v01c4's entire concept layer was destroyed because only
    # the reshaped result was ever written -- 90 calls, unrecoverable. Written pre-parse so
    # a malformed response is captured too, which is precisely when the original matters.
    raw_dir = Path(out_dir) / "_raw"
    raw_dir.mkdir(exist_ok=True)
    (raw_dir / f"{stem}.json").write_text(txt, encoding="utf-8")
    obj = _loads(txt)
    if not (isinstance(obj, dict) and isinstance(obj.get("Assets"), list)):
        raise RuntimeError(f"{stem}: unparseable asset response -- nothing cached")
    # PRESERVE EVERY TOP-LEVEL KEY. This used to be {"IP": stem, "Assets": obj["Assets"]},
    # which silently deleted anything else the model returned before it reached disk. The
    # T-1 arm asked for a "ConceptualAssets" array, the model emitted it, and this line
    # threw it away in all 90 files of v01c4 -- which was then read as the model refusing
    # to follow the contract. It also produced the false clue that broke the diagnosis:
    # obj["Assets"] passes through wholesale, so fields nested INSIDE each asset survived
    # while top-level keys vanished, and that asymmetry looked like evidence about the
    # model when it was evidence about this line.
    # Only IP is normalised, which is all the rebuild was ever for -- it must match the
    # filename. Unexpected keys are now cached rather than dropped, so the validators can
    # flag them instead of them disappearing without trace.
    result = {"IP": stem, **{k: v for k, v in obj.items() if k != "IP"}}
    outp.write_text(json.dumps(result, indent=2))
    return result

def _lev(a, b, cap=6):
    """Levenshtein distance, with an early exit once it exceeds `cap`.

    Only used to answer "is this a near-miss or an invention?", so a distance of 7 and a
    distance of 40 are the same answer and there is no reason to compute either exactly.
    """
    if a == b:
        return 0
    if abs(len(a) - len(b)) > cap:
        return cap + 1
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        if min(cur) > cap:
            return cap + 1
        prev = cur
    return prev[-1]


def validate_primary(result, parsed, rtl_text=""):
    """-> [record], one per problem. Keyed by (entity, name): duplicate names across a
    file's entities (e.g. clk_i in neorv32_trng/neoTRNG/neoTRNG_cell) must not yield false
    Entity mismatches.

    RECORDS, NOT STRINGS. Every version through v01c6p1 receives the parsed closed set and
    sits at 0 ungrounded names across all 90 module-repeats, so the string form cost
    nothing. P-2 withholds that list and asks the model to read the declarations out of the
    RTL, which makes an ungrounded name the arm's most informative output -- and it has
    three quite different causes that a string cannot distinguish:

      in_rtl_base True,  distance small  -> the element is real; our regex parser missed it,
                                            or the model wrote a variant spelling
      in_rtl_base True,  distance large  -> real identifier, wrong granularity or wrong name
      in_rtl_base False                  -> invented

    Only the first is a finding about rtl_parse rather than about the model, and separating
    them is the whole point of keeping this data.

    `rtl_text` is optional so the function still works from analysis scripts that have no
    source to hand; the RTL-presence fields are then None rather than False, because
    "not checked" and "checked, absent" must never look alike.
    """
    elems = parsed["ports"] + parsed["signals"]
    keys  = {(e["entity"], e["name"]) for e in elems}
    ents  = {}
    for e in elems:
        ents.setdefault(e["name"], set()).add(e["entity"])
    names = list(ents)
    OBJ = {"Confidentiality", "Integrity", "Availability"}
    out = []
    for a in result.get("Assets", []):
        r, en = a.get("Asset RTL"), a.get("Entity")
        obj = a.get("Security Objective")
        base = {"name": r, "entity_claimed": en, "concept": a.get("Concept"),
                "objective": obj}
        if r not in ents:
            tok = str(r).split(".")[0] if r else ""
            if rtl_text and tok:
                full = bool(re.search(rf"\b{re.escape(str(r))}\b", rtl_text))
                bse  = bool(re.search(rf"\b{re.escape(tok)}\b", rtl_text))
            else:
                full = bse = None
            near, dist = min(((n, _lev(str(r), n)) for n in names),
                             key=lambda t: t[1]) if names else (None, None)
            out.append({**base, "kind": "ungrounded", "in_rtl_full": full,
                        "in_rtl_base": bse, "nearest": near, "distance": dist,
                        "detail": f"ungrounded Asset RTL '{r}'"})
        elif (en, r) not in keys:
            out.append({**base, "kind": "entity_mismatch", "entity_actual": sorted(ents[r]),
                        "detail": f"Entity mismatch for '{r}': '{en}' not in {sorted(ents[r])}"})
        if obj not in OBJ:
            out.append({**base, "kind": "bad_objective",
                        "detail": f"bad objective for '{r}': '{obj}'"})
    return out


v01c6p1p2: core 7058 + ICL 62848 = 69908 chars   [expects ConceptualAssets]
v01c6p1p2: validation is REPORT-ONLY (no corrective regeneration) -- this version does not receive the parsed closed set


In [21]:
# ===== Stage B run loop: REPEATS x (generate -> validate -> one corrective retry) =====
# Each repeat re-runs the SAME prompt into its own directory. The spread across repeats
# is the noise floor; read it before believing any difference between versions.
# Re-running this cell is cheap: a module already cached in a repeat dir is not regenerated.
#
# Modules within a repeat run CONCURRENTLY; repeats stay sequential. Lower MAX_WORKERS if
# the API starts rate-limiting -- the client already retries with exponential backoff, but
# retries cost wall clock, so throttling here is cheaper than being throttled there.
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter
MAX_WORKERS = 6
for rep in range(REPEATS):
    out_dir = asset_dir(VERSION, rep, create=True)
    # Provenance: which prompt actually produced this directory. Underscore-prefixed so
    # eval_assets.load_run() skips it rather than reading it as a module.
    #
    # Written ONCE per directory, and never blindly. generate_assets() returns any module
    # already cached, so re-running into an existing directory keeps the old files and
    # generates only what is missing. If the prompt changed in between, that directory then
    # holds output from TWO prompts -- and overwriting the meta would stamp it with the new
    # sha and hide the mix completely. So:
    #   sha matches  -> leave the existing file alone; its timestamp is the true one
    #   sha differs  -> refuse; the directory can no longer be attributed to one prompt
    # (An earlier unconditional write clobbered v01c3 r0-r2's timestamps during the n=5
    #  extension. Harmless there only because the sha was unchanged.)
    meta_p = out_dir / "_run_meta.json"
    sha = hashlib.sha256(ASSET_PRIMARY_SYSTEM.encode()).hexdigest()
    if meta_p.exists():
        old = json.loads(meta_p.read_text())
        if old.get("system_prompt_sha256") != sha:
            raise RuntimeError(
                f"{out_dir} holds assets from prompt {str(old.get('system_prompt_sha256'))[:12]} "
                f"but the current prompt is {sha[:12]}. Re-running here would mix two prompts "
                f"in one run directory. Delete the directory, or use a new VERSION.")
    else:
        meta_p.write_text(json.dumps({
            "version": VERSION, "repeat": rep, "model": GEN_MODEL, "effort": "high",
            "system_prompt_sha256": sha,
            "system_prompt_chars": len(ASSET_PRIMARY_SYSTEM),
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        }, indent=2))

    unresolved = {}
    print(f"\n===== {VERSION} repeat {rep}  ->  {out_dir} =====")

    def _check(stem, res, path):
        """Every validation record for one result: structural + concept-contract.

        The RTL is re-read here rather than threaded down from generate_assets, so that a
        module served from cache is validated against the same source a freshly generated
        one is. It is a file read and a regex per module per repeat.
        """
        rtl_txt = rtl_parse.strip_comments(
            Path(path).read_text(encoding="utf-8", errors="ignore"),
            vhdl=Path(path).suffix.lower() in (".vhd", ".vhdl"))
        return (validate_primary(res, parsed_all[stem], rtl_txt)
                + [{"kind": "concept_contract", "name": None, "detail": s}
                   for s in DG.validate_concepts(res, EXPECT_CONCEPTS)])

    def _one(stem, path):
        """generate -> validate -> (only if RETRY_ON_VALIDATION) one corrective retry."""
        res = generate_assets(stem, path, parsed_all[stem], out_dir)
        recs = _check(stem, res, path)
        note = ""
        if recs and RETRY_ON_VALIDATION:
            note = f"[warn] {stem}: {len(recs)} issue(s) -> one regeneration\n        " \
                   + "\n        ".join(r["detail"] for r in recs)
            res2 = generate_assets(stem, path, parsed_all[stem], out_dir, overwrite=True)
            recs2 = _check(stem, res2, path)
            if len(recs2) <= len(recs):
                res, recs = res2, recs2
            else:   # first attempt was better -- restore it to the cache
                (out_dir / f"{stem}.json").write_text(json.dumps(res, indent=2))
        return res, recs, note

    # The 18 modules are independent, I/O-bound calls that each write their own file, so
    # they parallelise cleanly. Wall clock was ~3.9 min/call sequentially; the work is
    # almost entirely waiting on the API, not local compute (which is also why a GPU does
    # nothing here -- the model runs on OpenAI's side).
    # Results are collected and printed IN MODULE ORDER after the pool drains, so the log
    # stays diffable across runs instead of interleaving by completion time.
    done = {}
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futs = {pool.submit(_one, stem, path): stem for stem, path in modules}
        for fut in as_completed(futs):
            stem = futs[fut]
            try:
                done[stem] = fut.result()
            except Exception as e:
                done[stem] = (None, None, f"[err ] {stem:24s} {e}")

    # The report is WRITTEN, not just printed. It used to live in a local `unresolved`
    # dict that died with the cell, which was tolerable while every version sat at zero
    # issues. Underscore-prefixed so eval_assets.load_run() skips it -- it globs "*.json"
    # and ignores names starting with "_", so this cannot be scored as a module.
    report = {"version": VERSION, "repeat": rep, "model": GEN_MODEL,
              "system_prompt_sha256": sha,
              "retry_on_validation": RETRY_ON_VALIDATION,
              "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
              "modules": {}, "totals": {}}
    for stem, _path in modules:
        res, recs, note = done[stem]
        if note:
            print(note)
        if res is None:
            continue
        if recs:
            unresolved[stem] = [r["detail"] for r in recs]
        kinds = Counter(r["kind"] for r in recs)
        report["modules"][stem] = {
            "assets": len(res.get("Assets", [])),
            "concepts": len(res.get("ConceptualAssets", []) or []),
            "by_kind": dict(kinds),
            "issues": recs,
        }
        print(f"[ok  ] {stem:24s} {len(res['Assets']):3d} assets"
              + (f"   {dict(kinds)}" if recs else ""))

    tot = Counter()
    for _m in report["modules"].values():
        tot.update(_m["by_kind"])
        tot["assets"] += _m["assets"]
    report["totals"] = dict(tot)
    (out_dir / "_validation.json").write_text(json.dumps(report, indent=1), encoding="utf-8")

    _n = tot.get("assets", 0)
    print(f"  validation -> {out_dir / '_validation.json'}")
    print(f"  totals: {dict(tot)}"
          + (f"   ungrounded {tot.get('ungrounded', 0) / _n:.1%} of emissions" if _n else ""))
    print("  policy: " + ("REPORT-ONLY -- no asset was regenerated on a validation result"
                          if not RETRY_ON_VALIDATION else
                          f"retry ON -- unresolved: {unresolved or 'none'}"))

print("\n", client.cost())



===== v01c6p1p2 repeat 0  ->  assets_tuning18_v01c6p1p2_r0 =====
[ok  ] neorv32_boot_rom           5 assets   {'ungrounded': 1}
[ok  ] neorv32_bus               50 assets
[ok  ] neorv32_cache             44 assets
[ok  ] neorv32_cpu               34 assets
[ok  ] neorv32_cpu_cp_cfu        22 assets
[ok  ] neorv32_cpu_cp_muldiv     26 assets
[ok  ] neorv32_cpu_pmp           24 assets
[ok  ] neorv32_debug_dtm         20 assets
[ok  ] neorv32_fifo              25 assets
[ok  ] neorv32_hwspinlock         4 assets
[ok  ] neorv32_imem              11 assets   {'ungrounded': 1}
[ok  ] neorv32_package           38 assets   {'ungrounded': 38}
[ok  ] neorv32_spi               48 assets
[ok  ] neorv32_sys               15 assets
[ok  ] neorv32_trng              17 assets
[ok  ] neorv32_twi               45 assets
[ok  ] neorv32_uart              43 assets
[ok  ] neorv32_wdt               22 assets
  validation -> assets_tuning18_v01c6p1p2_r0\_validation.json
  totals: {'ungrounded': 40, 'assets'

# Stage C: Evaluation

Scores generated asset lists against two references built from the LAsset release:

* **`gt`** - `Asset_Dataset_Statistics_NEORV32.xlsx`, sheet `Assets (Manual)`. The paper's
  manually annotated "True Assets". This is ground truth.
* **`paper`** - `asset_list_neorv32_initial.json`, the output of the paper's own generation
  stage. Not ground truth; it is the target to match, and it has its own false positives.

**Metrics.** TP / FP / FN with precision, recall and F1 - the columns the paper's statistics
sheet uses. No TN and no FPR: the negative class ("every parsed element that is not an
asset") outnumbers the positives about 14:1, so FPR reads ~0.1 while precision is ~0.3.

**Matching is strict.** A prediction is credited only on an exact name, or on the one
legitimate near-match where the reference names a whole record and the prediction names one
of its fields (or vice versa). Field-to-field matching is refused: crediting a predicted
`fifo.avail` against a ground-truth `fifo.re` rewards spraying record fields, and it inflated
recall unevenly across versions (v0 by 4 TP, v1 by 2) - which biases the very comparison the
ablation exists to make. Pass `strict=False` to reproduce the old numbers.

**Reading the ablation table.** The `sd` columns come from the repeats and are the noise
floor. Any difference between two versions smaller than their sd is not measurable. The
paired tests below the table use modules as the pairing unit - each module is its own
control, which matters because between-module variance is large (cpu ground-truth density
~38%, peripherals ~7%). A CI that straddles zero is an honest null result.

**Known reference limits.** One ground-truth element is unreachable: `neorv32_cache.inval_i`
does not exist in `RTL_data` (v1.11.4.3), where the port is named `inv_i` - the reference was
annotated against a different NEORV32 revision. A second element is unreachable for the same
reason: `cache_o.cmd_dir`, whose record carries cmd_clr / cmd_inv / cmd_new and no cmd_dir
anywhere in the file. Max achievable recall is therefore 0.982, and neorv32_cache's own
ceiling is 4/6 = 0.667 - read that module's recall against 0.667, not 1.0. (Audited
2026-08-07; the earlier 0.991 counted only inval_i.)


In [22]:
# ===== Build the reference sets from the LAsset release (run once; re-run if the xlsx changes)
import importlib, gt_extract
importlib.reload(gt_extract)

LASSET_ROOT = Path(r"E:\jobs\ff\test\LAsset-Security-Assets")   # <-- the published results repo
GT_DIR      = Path("ground_truth")

gt_extract.main(LASSET_ROOT, GT_DIR)

# Sanity: the extracted counts must reconcile with the paper's own Statistics sheet
# (302 True Assets / 359 initial / 331 refined). A mismatch means the sheet layout
# changed and IP_LABEL_TO_STEM or split_elements() needs revisiting.
for f in sorted(GT_DIR.glob("*.json")):
    d = json.loads(f.read_text(encoding="utf-8"))
    n = sum(len(m["assets"]) for m in d["modules"].values())
    print(f"  {f.name:28s} {len(d['modules']):3d} modules, {n:4d} elements")

manual GT : 41 modules, 246 rows -> 302 elements -> ground_truth\manual_gt_neorv32.json
            (paper's Statistics sheet reports 302 True Assets)
lasset initial : 41 modules, 359 assets -> ground_truth\lasset_initial.json
lasset refined : 41 modules, 331 assets -> ground_truth\lasset_refined.json
lasset final   : empty, skipped
  lasset_initial.json           41 modules,  359 elements
  lasset_refined.json           41 modules,  331 elements
  manual_gt_neorv32.json        41 modules,  302 elements


In [23]:
# ===== Score one run of the current VERSION =====
import importlib, eval_assets as EV
importlib.reload(EV)

REFS = EV.load_refs(GT_DIR)          # {"gt", "paper", "paper_refined"}

# collect() finds the repeat directories, and falls back to a pre-repeats run directory.
_runs = EV.collect([VERSION])
res = EV.score(_runs[VERSION][0], REFS["gt"]) if VERSION in _runs else None
if res:
    EV.table(res, f"{VERSION} vs manual ground truth (first repeat)")
else:
    print(f"no generated assets for {VERSION} yet -- run Stage B first")


v01c6p1p2 vs manual ground truth (first repeat)
                        ref  emit   TP   FP   FN      P      R     F1    obj
----------------------------------------------------------------------------
bus                      11    50   11   39    0  0.220  1.000  0.361  1.000
cache                     6    44    4   40    2  0.091  0.667  0.160  0.750
cpu                      14    34   13   21    1  0.382  0.929  0.542  0.923
cpu_cp_cfu               11    22    9   13    2  0.409  0.818  0.545  0.667
cpu_cp_muldiv            12    26   12   14    0  0.462  1.000  0.632  0.750
cpu_pmp                   6    24    6   18    0  0.250  1.000  0.400  0.667
debug_dtm                 5    20    3   17    2  0.150  0.600  0.240  1.000
hwspinlock                2     4    2    2    0  0.500  1.000  0.667  1.000
imem                      3    11    3    8    0  0.273  1.000  0.429  0.333
spi                       8    48    8   40    0  0.167  1.000  0.286  1.000
sys                       2 

In [27]:
# ===== ABLATION: every version, mean +/- sd over repeats =====
# Read the sd columns FIRST. They are the noise floor: a gap between two versions smaller
# than their sd is not measurable at this many repeats, however suggestive it looks.
importlib.reload(EV)

VERSIONS = ["v0", "v01", "v01c2", "v01c3", "v01r", "v01c4", "v01c5", "v01c6",
            "v01c6p1", "v01c6p1p2"]   # add once run

# Each arm has a DESIGNATED baseline -- the version it differs from by one variable.
# Pairing everything against VERSIONS[0] silently produces multi-variable contrasts:
# v01c2 vs v0 moves the core prompt AND the ICL block, and is not an arm at all.
BASELINE = {"v01": "v0", "v02": "v01", "v01c2": "v01", "v01c3": "v01c2",
            "v01r": "v01", "v01c4": "v01c3", "v01c5": "v01c4",
            "v01c6": "v01c5", "v01c6p1": "v01c6", "v01c6p1p2": "v01c6p1"}
# A new arm with no explicit entry pairs against the standing baseline. Explicit
# entries above always win, so promoting CURRENT_BASELINE can never silently
# re-point an old arm at a comparison it was not run against.
for _v in VERSIONS:
    if _v not in BASELINE and _v != CURRENT_BASELINE and _v != VERSIONS[0]:
        BASELINE[_v] = CURRENT_BASELINE

RUNS = EV.collect(VERSIONS)               # {version: [run per repeat]}; missing ones warn

if RUNS:
    # The paper's own generation stage, on exactly the modules every run has in common --
    # same algorithmic stage as ours, so this row is the target, not the ground truth.
    MODS = EV.common_modules(RUNS, REFS["gt"])
    paper_here = {m: v for m, v in REFS["paper"].items() if m in MODS}

    EV.ablate(RUNS, REFS["gt"], extra={"LAsset paper": paper_here})

    # PER-CLASS paired deltas are the primary result, not the aggregate. A-01 is why:
    # aggregate recall read +0.049 [-0.061, +0.166] -- "no effect" -- while port recall
    # rose +0.181 [+0.048, +0.302] and signal recall fell -0.157 [-0.229, -0.086]. Both
    # class effects clear zero; they cancel in the aggregate. Reporting only the aggregate
    # would have thrown away a resolvable result and called the arm a null.
    for v in VERSIONS:
        base = BASELINE.get(v)
        if v in RUNS and base in RUNS:
            n = min(len(RUNS[v]), len(RUNS[base]))
            note = "  [DIRECTIONAL ONLY -- fewer than 3 repeats, no usable sd]" if n < 3 else ""
            print(f"\n--- {v} vs its baseline {base}{note}")
            EV.paired_classes(RUNS, base, v, REFS["gt"])
        elif v in RUNS and base:
            print(f"\n--- {v}: baseline {base!r} has no run yet, skipped")
else:
    print("no runs found -- run Stage B for at least one version first")


ablation vs manual ground truth
scored on 15 modules common to every run
version       n  emit   TP   FP   FN      P  recall     sd      F1     sd  near
-------------------------------------------------------------------------------
v0            3   205   61  143   50  0.300   0.553  0.019   0.389  0.010   1.7
v01           3   235   67  167   44  0.288   0.607  0.005   0.390  0.015   1.3
v01c2         5   297   83  215   28  0.277   0.744  0.069   0.404  0.024   1.2
v01c3         5   274   87  187   24  0.318   0.784  0.059   0.452  0.025   0.2
v01r          2   210   70  140   42  0.331   0.626  0.032   0.433  0.009   0.0
v01c4         5   352   94  258   17  0.267   0.845  0.024   0.406  0.009   0.2
v01c5         5   331   94  237   17  0.285   0.849  0.017   0.426  0.004   0.4
v01c6         5   428   98  330   13  0.229   0.881  0.021   0.364  0.021   0.4
v01c6p1       5   476  100  376   11  0.210   0.897  0.015   0.340  0.017   0.2
v01c6p1p2     3   404   98  306   13  0.243   0

In [25]:
# ===== STAGE C SUBSTAGE -- T-1: conception failure vs binding failure =====
# DIAGNOSTIC ONLY. Classifies MISSES; it is not a metric and no arm is accepted or rejected
# on it. v01c4 changes the output contract and asks for reasoning that was internal, either
# of which can move behaviour on its own -- so its recall delta is CONFOUNDED, and asking
# for the concept list BEFORE the assets may itself cost recall by forcing early commitment.
import importlib, diagnose_step1 as DG
importlib.reload(DG)
from collections import Counter

# Pointing this at the wrong version is not a harmless mistake: the first run of this cell
# analysed v01c4 -- the one version whose concepts generate_assets() had stripped -- and
# reported 100% "conception failure" while v01c5 sat on disk fully populated. The guard
# below refuses to analyse a run that has no concept list anywhere, because the only
# output such a run can produce is a confident wrong answer.
DIAG_VERSION = "v01c6"
_dirs = sorted(Path(".").glob(f"assets_tuning18_{DIAG_VERSION}_r*"))
if _dirs and not any(v["concepts"] for v in DG.load_concept_runs(_dirs[0]).values()):
    raise RuntimeError(
        f"{DIAG_VERSION} r0 has no ConceptualAssets in any module. Either the prompt did "
        f"not ask for them, or generate_assets() stripped them. Analysing this would "
        f"report 100% conception failure, and that would be an artefact.")

if not _dirs:
    print(f"no {DIAG_VERSION} runs yet -- set VERSION = '{DIAG_VERSION}' and run Stage B")
else:
    _runs = EV.collect([DIAG_VERSION, "v01c3"])
    _mods = EV.common_modules(_runs, REFS["gt"])

    # The judge. Blind by construction: it never learns whether an element was found or
    # missed, nor which concept the model itself linked it to.
    def _call(system, user):
        # effort='low': the judge answers with one token (an id or NONE), and reasoning
        # shares the output budget. 2048 is a ceiling, not a reservation.
        return client.complete_text(system, user, max_tokens=2048, effort="low")
    JUDGE = DG.make_judge(_call)

    # ---- CALIBRATION FIRST. Not optional: with a judge that always picks a concept the
    # split reads 100% (b), with one that always says NONE it reads 100% (a). The `any`
    # rate bounds every (a) verdict, because a judge that answers NONE on links the model
    # explicitly made will answer NONE on misses too.
    _cal = DG.calibrate_judge(DG.load_concept_runs(_dirs[0]), REFS["gt"],
                              mods=_mods, judge=JUDGE, limit=40)
    print(f"JUDGE CALIBRATION on {_cal['n']} FOUND references (model asserted the link):")
    print(f"  exact concept id agreement : {_cal['exact']:.2f}")
    print(f"  any-concept agreement      : {_cal['any']:.2f}   <- bounds the (a) verdicts")
    if _cal["any"] < 0.80:
        print(f"  [warn] judge answers NONE on {100*(1-_cal['any']):.0f}% of links the model "
              f"MADE. Every (a) below inherits that false-negative rate.")

    # ---- PRIMARY SPLIT, over DISTINCT references per repeat.
    agg, K = Counter(), len(_dirs)
    for _d in _dirs:
        _rows, _t = DG.split_misses(DG.load_concept_runs(_d), REFS["gt"],
                                    mods=_mods, judge=JUDGE)
        agg.update(_t)
    n = sum(agg.values())
    print(f"\n(a)/(b) SPLIT -- {n/K:.1f} distinct misses per repeat, {K} repeats")
    for k in DG.VERDICTS:
        if agg[k]:
            print(f"  {agg[k]/K:6.1f}/repeat ({100*agg[k]/n:5.1f}%)  {k}")
    print("  (b) is sound; (a) is weaker -- absence from the emitted list is not proof the")
    print("  model never conceived it. Do not present them as symmetric.")

    # ---- CROSS-CHECK. Disagreements are the rows worth reading by hand.
    _cr, _cor = DG.corroborating_evidence(EV.load_run(_dirs[0]), REFS["gt"], _mods)
    print("\ncorroboration on r0 (sibling / secondary, NOT verdicts):")
    for k in DG.TIERS:
        if _cor[k]:
            print(f"  {_cor[k]:4d}  {k}")

    # ---- Q5: are FPs over-binding of legitimate concepts, or bad concepts?
    _out, _per = DG.concept_precision(DG.load_concept_runs(_dirs[0]), REFS["gt"], _mods)
    print("\nconcept-level precision (r0):")
    for k, v in _out.most_common():
        print(f"  {v:4d}  {k}")

    # ---- UNBOUND CONCEPTS: a third outcome, distinct from (a) and (b).
    # The model conceived an idea and bound no element to it. The prompt explicitly invites
    # this ("a conceptual asset with no closed-set element still belongs here"), so it is
    # not a contract violation -- it is the model reporting that the closed set cannot
    # express something it considered. cache.inval_i is the known case: class `absent`,
    # genuinely not in the parsed set because of the C-03 revision skew.
    _ub, _rows = DG.concept_binding(DG.load_concept_runs(_dirs[0]), _mods)
    print("\nconcept binding (r0):")
    for k, v in _ub.most_common():
        print(f"  {v:4d}  {k}")
    if _rows:
        print("  unbound concepts, first 8:")
        for r in _rows[:8]:
            print(f"    {r['module'].replace('neorv32_',''):16s} {r['id']:4s} "
                  f"[{r['objective'] or '?'}] {r['concept'][:58]}")


JUDGE CALIBRATION on 40 FOUND references (model asserted the link):
  exact concept id agreement : 0.68
  any-concept agreement      : 1.00   <- bounds the (a) verdicts

(a)/(b) SPLIT -- 13.2 distinct misses per repeat, 5 repeats
    10.6/repeat ( 80.3%)  (b) conceived but unbound -- a concept covers it
     1.6/repeat ( 12.1%)  (a) not conceived -- objective absent from the concept list
     1.0/repeat (  7.6%)  (a) not conceived -- judge finds no covering concept
  (b) is sound; (a) is weaker -- absence from the emitted list is not proof the
  model never conceived it. Do not present them as symmetric.

corroboration on r0 (sibling / secondary, NOT verdicts):
     5  sibling: concept reached (b)
     6  secondary: concept reached (b)
     3  unresolved -> judge
     2  solo, nothing emitted nearby

concept-level precision (r0):
   170  FP under a concept that also produced a TP (over-binding)
   152  FP under a concept that produced no TP (bad concept)

concept binding (r0):
    96  

In [26]:
# ===== What this version missed and what it invented =====
if res:
    EV.table(res, f"{VERSION} vs manual GT", show_names=True)


v01c6p1p2 vs manual GT
                        ref  emit   TP   FP   FN      P      R     F1    obj
----------------------------------------------------------------------------
bus                      11    50   11   39    0  0.220  1.000  0.361  1.000
cache                     6    44    4   40    2  0.091  0.667  0.160  0.750
cpu                      14    34   13   21    1  0.382  0.929  0.542  0.923
cpu_cp_cfu               11    22    9   13    2  0.409  0.818  0.545  0.667
cpu_cp_muldiv            12    26   12   14    0  0.462  1.000  0.632  0.750
cpu_pmp                   6    24    6   18    0  0.250  1.000  0.400  0.667
debug_dtm                 5    20    3   17    2  0.150  0.600  0.240  1.000
hwspinlock                2     4    2    2    0  0.500  1.000  0.667  1.000
imem                      3    11    3    8    0  0.273  1.000  0.429  0.333
spi                       8    48    8   40    0  0.167  1.000  0.286  1.000
sys                       2    15    2   13    0  0.1